# Day 13 - Validating CSV files

All ten tasks in one notebook. Use **Run All** from the top.

## Settings and rules (tasks 1, 5, 6, 10)

In [ ]:
from pathlib import Path
import csv
import math
import re
from datetime import datetime

# Settings: change them here, not inside the functions
INPUT_DIR = Path("olist_data")
BAD_RECORDS_FILE = Path("bad_records.csv")
MAX_REJECT_RATE = 0.20   # Task 10: fail a whole file if more than 20% of its rows are rejected

FILES_TO_VALIDATE = ["olist_orders_dataset.csv", "order_items.csv", "customers.csv"]

# Task 1: required fields for each file
REQUIRED_FIELDS = {
    "olist_orders_dataset.csv": ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
    "order_items.csv": ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"],
    "customers.csv": ["customer_id", "customer_zip_code_prefix"],
}

# Task 5: allowed order statuses (business rule)
ALLOWED_STATUSES = {"approved", "created", "processing", "invoiced", "shipped", "delivered", "unavailable", "canceled"}

# Task 6: regular expressions
ZIP_PATTERN = re.compile(r"[0-9]{5}")          # zip prefix: exactly 5 digits
ID_PATTERN = re.compile(r"[0-9a-fA-F]{32}")    # IDs: exactly 32 hex characters
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

print("Settings loaded. Reading files from:", INPUT_DIR)


## Task 7: custom exceptions

In [ ]:
class SchemaError(Exception):
    """A file is missing required columns."""


class FileFormatError(Exception):
    """A file is empty, has no header row, or is not valid UTF-8."""


## Tasks 2, 7, 8: read a file and check its columns

In [ ]:
def read_rows(path: Path) -> tuple[list[str], list[dict]]:
    """Read a CSV file and return (header, rows).

    Raises FileFormatError if the file is empty, has no header, or is not UTF-8.
    """
    try:
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            header = reader.fieldnames
            rows = list(reader)
    except UnicodeDecodeError as err:
        raise FileFormatError(f"{path.name} is not valid UTF-8: {err}") from err
    if not header:
        raise FileFormatError(f"{path.name} is empty or has no header row")
    return header, rows


def check_schema(filename: str, header: list[str]) -> None:
    """Task 1 and 7: raise SchemaError if a required column is missing."""
    missing = [col for col in REQUIRED_FIELDS[filename] if col not in header]
    if missing:
        raise SchemaError(f"{filename} is missing required columns: {missing}")


## Tasks 4, 5, 6: type, business rule and regex checks
Each check returns `None` when the value is fine, or a reason when it is not.

In [ ]:
def check_id(value: str) -> str | None:
    return None if ID_PATTERN.fullmatch(value) else "must be 32 hex characters"


def check_zip(value: str) -> str | None:
    return None if ZIP_PATTERN.fullmatch(value) else "zip prefix must be 5 digits"


def check_status(value: str) -> str | None:
    return None if value in ALLOWED_STATUSES else f"status '{value}' is not allowed"


def check_timestamp(value: str) -> str | None:
    try:
        datetime.strptime(value, DATE_FORMAT)
        return None
    except ValueError:
        return f"'{value}' is not a date in the format YYYY-MM-DD HH:MM:SS"


def check_price(value: str) -> str | None:
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    if not math.isfinite(number):
        return f"'{value}' is not a finite number"
    return None if number > 0 else "price must be greater than 0"


def check_freight(value: str) -> str | None:
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    if not math.isfinite(number):
        return f"'{value}' is not a finite number"
    return None if number >= 0 else "freight must be 0 or more"


# Which check applies to which column
FIELD_CHECKS = {
    "order_id": check_id,
    "customer_id": check_id,
    "product_id": check_id,
    "seller_id": check_id,
    "customer_zip_code_prefix": check_zip,
    "order_status": check_status,
    "order_purchase_timestamp": check_timestamp,
    "order_delivered_customer_date": check_timestamp,
    "price": check_price,
    "freight_value": check_freight,
}


## Task 2: validate every row and collect all errors

In [ ]:
def validate_row(row: dict, filename: str) -> list[str]:
    """Return every problem in one row. An empty list means the row is valid."""
    errors = []
    for col in REQUIRED_FIELDS[filename]:
        if (row.get(col) or "").strip() == "":
            errors.append(f"{col} is blank")
    for col, check in FIELD_CHECKS.items():
        value = (row.get(col) or "").strip()
        if value == "":
            continue                     # blanks are handled by the required-field check
        problem = check(value)
        if problem:
            errors.append(f"{col}: {problem}")
    return errors


def validate_file(filename: str) -> dict:
    """Read one file, check every row, and collect the rejected rows."""
    path = INPUT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"{filename} not found in {INPUT_DIR}")
    header, rows = read_rows(path)
    check_schema(filename, header)

    valid_count = 0
    rejected = []
    for line_no, row in enumerate(rows, start=2):    # line 1 is the header
        errors = validate_row(row, filename)
        if errors:
            record = "; ".join(f"{k}={v or ''}" for k, v in row.items() if k is not None)
            rejected.append({
                "file": filename,
                "line": line_no,
                "reason": "; ".join(errors),
                "record": record,
            })
        else:
            valid_count += 1

    return {"file": filename, "total": len(rows), "valid_count": valid_count,
            "rejected": rejected, "status": "OK"}


def apply_reject_threshold(result: dict) -> None:
    """Task 10: mark the whole file FAILED if too many of its rows were rejected."""
    if result["total"] == 0:
        return
    rate = len(result["rejected"]) / result["total"]
    if rate > MAX_REJECT_RATE:
        result["status"] = f"FAILED: {rate:.0%} rejected, limit is {MAX_REJECT_RATE:.0%}"


## Run validation on every file (tasks 7, 8, 10)
A missing file is skipped and a bad file fails only itself. The run continues either way.

In [ ]:
all_results = []
for filename in FILES_TO_VALIDATE:
    try:
        result = validate_file(filename)
        apply_reject_threshold(result)
    except FileNotFoundError as err:
        # Task 8: a missing file is skipped
        result = {"file": filename, "total": 0, "valid_count": 0, "rejected": [],
                  "status": f"SKIPPED: {err}"}
    except (SchemaError, FileFormatError) as err:
        # Task 7: a bad schema or unreadable file fails that file only
        result = {"file": filename, "total": 0, "valid_count": 0, "rejected": [],
                  "status": f"FAILED: {err}"}
    all_results.append(result)

print("Validation finished for", len(all_results), "files")


## Task 9: summary of total, valid and rejected rows

In [ ]:
def print_summary(all_results: list[dict]) -> None:
    """Print total, valid and rejected rows for each file, then overall."""
    print(f"{'File':<30}{'Total':>8}{'Valid':>8}{'Rejected':>10}  Status")
    print("-" * 90)
    for r in all_results:
        print(f"{r['file']:<30}{r['total']:>8}{r['valid_count']:>8}{len(r['rejected']):>10}  {r['status']}")
    total = sum(r["total"] for r in all_results)
    valid = sum(r["valid_count"] for r in all_results)
    rejected = sum(len(r["rejected"]) for r in all_results)
    print("-" * 90)
    print(f"Overall: total={total}, valid={valid}, rejected={rejected}")


print_summary(all_results)


## Task 3: write rejected rows, with their reasons, to bad_records.csv

In [ ]:
def write_bad_records(all_results: list[dict], path: Path = BAD_RECORDS_FILE) -> int:
    """Write every rejected row, with its reason, to a CSV file. Return the count."""
    rejected = [rec for result in all_results for rec in result["rejected"]]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "line", "reason", "record"])
        writer.writeheader()
        writer.writerows(rejected)
    return len(rejected)


n = write_bad_records(all_results)
print(f"Wrote {n} rejected rows to {BAD_RECORDS_FILE}")

# Preview the first lines of the file
for line in BAD_RECORDS_FILE.read_text(encoding="utf-8").splitlines()[:5]:
    print(line)


## Demo: each rule on its own (tasks 4, 5, 6, 7, 8)

In [ ]:
def show(check, value):
    result = check(value)
    print(f"{value!r:<45} -> {result or 'OK'}")


print("Task 6, regex:")
show(check_zip, "14409")
show(check_zip, "1046")
show(check_id, "5d835b501c8e0eb0ce83c72a17cb101a")
show(check_id, "not-a-valid-id")

print("\nTask 4, types and dates:")
show(check_price, "19.18")
show(check_price, "abc")
show(check_timestamp, "2018-03-04 12:48:00")
show(check_timestamp, "2018-02-30 10:00:00")

print("\nTask 5, business rules:")
show(check_price, "0")
show(check_status, "delivered")
show(check_status, "lost")

print("\nTask 7, custom exception:")
try:
    check_schema("customers.csv", ["customer_id"])
except SchemaError as err:
    print("SchemaError:", err)

print("\nTask 8, missing file:")
try:
    validate_file("does_not_exist.csv")
except FileNotFoundError as err:
    print("Handled:", err)
